# Backtest con identificazione del giorno tramite modello

Questo notebook replica `targeted_experiment_run.ipynb` e aggiunge il classificatore salvato in `notebooks/models/attack_day/attack_day.pkl` come timing policy ulteriore.

Per ogni giorno candidato vengono costruite le feature disponibili **prima dell'attacco**; il giorno viene selezionato quando la probabilità stimata di attacco supera la soglia salvata in `metadata.json`. Il classificatore viene valutato per entrambi gli obiettivi (`force_buy` e `force_sell`), come nel notebook di riferimento.

In [ ]:
from pathlib import Path
import os
import json
import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path


PROJECT_ROOT = Path("/content/drive/MyDrive/master-thesis").resolve()

SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

os.chdir(PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(SRC_PATH))

from notebooks.targeted_experiment_utils import (
    ATSSetup,
    TARGET_SIGNALS,
    build_attack_day_plan,
    candidate_attack_days,
    compute_summary_by_setup,
    compute_summary_by_setup_and_timing,
    compute_timing_summaries,
    get_predictions_and_actuals,
    run_targeted_attack_experiment,
)
from scr.attack_day_model.model import load_model
from scr.attack_day_model.preprocessing import (
    build_attack_day_feature_row,
    encode_attack_day_features,
    engineer_features,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
logger = logging.getLogger('attack_day_backtest')

In [ ]:
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
MODEL_FOLDERS = {
    'LSTM': NOTEBOOK_DIR / 'models/LSTM_13_years',
    'LSTM3y': NOTEBOOK_DIR / 'models/LSTM_3_years',
    'LSTM_cumulative': NOTEBOOK_DIR / 'models/LSTM_cumulative',
    'LSTM_rolling': NOTEBOOK_DIR / 'models/LSTM_rolling',
    'transformer': NOTEBOOK_DIR / 'models/transformer',
}
DATA_FOLDER = PROJECT_ROOT / 'data/validation_time'
ATTACK_MODEL_FOLDER = NOTEBOOK_DIR / 'models/attack_day'
MODEL_PATH = ATTACK_MODEL_FOLDER / 'attack_day.pkl'
METADATA_PATH = ATTACK_MODEL_FOLDER / 'metadata.json'
RESULTS_DIR = NOTEBOOK_DIR / 'results_targeted/attack_day_model'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_START = '2013-01-01'
ATTACK_START = '2023-10-24'
ATTACK_END = '2026-06-01'
QUARTERLY_MONTHS = 3
EVALUATION_WINDOW_DAYS = 20
SEQUENCE_LENGTH = 50
OHLCV_WINDOW = 20
N_ATTACK_DAYS = 12
MAX_ABS_DELTA = 0.35
GRID_STEPS = 35
TIMING_POLICIES = [
    'uniform',
    'high_volatility',
    'high_pred_volatility',
    'strong_trend',
    'weak_trend',
    'near_strategy_boundary',
]
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ALL_TICKERS = [
    'GOOGL', 'AMZN', 'AAPL', 'PEP', 'JNJ', 'PFE', 'MRK', 'ABBV', 'PG', 'KO',
    'WMT', 'JPM', 'BAC', 'GS', 'V', 'XOM', 'CVX', 'COP', 'BP', 'BA',
    'MMM', 'HON', 'GE', 'T', 'VZ', 'TMUS', 'HSY', 'DUK', 'SO', 'EXC', 'AEP',
    'AMT', 'PLD', 'SPG', 'BHP', 'RIO', 'VALE', 'FCX',
]

SETUPS = [
    ATSSetup("LSTM_rolling_All_MA", tuple(ALL_TICKERS), "MA", model_name="LSTM_rolling", model_variant="quarterly", attacked_ticker="GOOGL"),
    ATSSetup("LSTM_cumulative_All_MA", tuple(ALL_TICKERS), "MA", model_name="LSTM_cumulative", model_variant="quarterly", attacked_ticker="GOOGL"),
    ATSSetup("LSTM3y_All_MA", tuple(ALL_TICKERS), "MA", model_name="LSTM3y", model_variant="single", attacked_ticker="GOOGL"),
    ATSSetup("LSTM13y_All_MA", tuple(ALL_TICKERS), "MA", model_name="LSTM", model_variant="single", attacked_ticker="GOOGL"),

]

TRAIN_MODE_BY_SETUP = {
    'LSTM_rolling_All_MA': 'rolling',
    'LSTM_cumulative_All_MA': 'cumulative',
    'LSTM3y_All_MA': 'cumulative',
    'LSTM13y_All_MA': 'cumulative',
}

with METADATA_PATH.open(encoding='utf-8') as f:
    ATTACK_MODEL_METADATA = json.load(f)
ATTACK_MODEL = load_model(MODEL_PATH, logger)
MODEL_THRESHOLD = float(ATTACK_MODEL_METADATA['threshold'])
MODEL_FEATURE_COLUMNS = ATTACK_MODEL_METADATA['feature_columns']
print(f'Model: {MODEL_PATH}')
print(f'Threshold: {MODEL_THRESHOLD:.3f} | features: {len(MODEL_FEATURE_COLUMNS)}')

## Costruzione delle predizioni del modello

Le righe sintetiche contengono solo informazioni note al giorno candidato. Il modello viene valutato per entrambi gli obiettivi del notebook di riferimento (`force_buy` e `force_sell`), così il confronto con le timing policy tradizionali resta omogeneo. Le colonne non presenti nell'input vengono aggiunte con 0 per mantenere esattamente l'ordine e lo schema del modello salvato. Per ogni coppia setup-obiettivo vengono scelti al massimo `N_ATTACK_DAYS` giorni tra quelli sopra soglia, ordinati per probabilità.

In [ ]:
def _model_scores_for_setup(setup):
    train_mode = TRAIN_MODE_BY_SETUP.get(setup.name, 'cumulative')
    days = candidate_attack_days(
        setup=setup, model_folders=MODEL_FOLDERS, project_root=PROJECT_ROOT,
        data_folder=DATA_FOLDER, sequence_length=SEQUENCE_LENGTH, min_day=30,
        simulation_start=ATTACK_START, simulation_end=ATTACK_END,
        data_start=DATA_START, months=QUARTERLY_MONTHS, train_mode=train_mode,
    )
    rows = []
    for day in days:
        for target_name in TARGET_SIGNALS:
            event = pd.Series({
                    'setup': setup.name, 'model': setup.model_name,
                    'strategy': setup.strategy_name, 'assets': ','.join(setup.tickers),
                    'attacked_ticker': setup.attacked_ticker, 'attack_day': int(day),
                    'objective': target_name, 'timing_policy': 'attack_day_model',
            })
            row = build_attack_day_feature_row(
                    event, price_data_folder=DATA_FOLDER,
                    simulation_start=ATTACK_START, simulation_end=ATTACK_END,
                    data_start=DATA_START, sequence_length=SEQUENCE_LENGTH,
                    ohlcv_window=OHLCV_WINDOW, prefer_download=False,
            )
            rows.append(row)

    raw = pd.DataFrame(rows)
    encoded, _ = encode_attack_day_features(raw)
    model_input = engineer_features(encoded)
    X = model_input.reindex(columns=MODEL_FEATURE_COLUMNS, fill_value=0.0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    probabilities = ATTACK_MODEL.predict_proba(X)[:, 1]
    scored = raw[['setup', 'attacked_ticker', 'attack_day', 'objective']].copy()
    scored['attack_probability'] = probabilities
    scored['selected_by_model'] = scored['attack_probability'] >= MODEL_THRESHOLD
    return scored.sort_values(['objective', 'attack_probability'], ascending=[True, False]).reset_index(drop=True)

def select_model_attack_days(scores, n_days=N_ATTACK_DAYS):
    # Come nel notebook tradizionale, il budget è N_ATTACK_DAYS per obiettivo.
    eligible = scores[scores['selected_by_model']].copy()
    selected = (
        eligible.sort_values('attack_probability', ascending=False)
        .groupby(['setup', 'objective'], group_keys=False)
        .head(n_days)
        .sort_values(['setup', 'objective', 'attack_day'])
        .reset_index(drop=True)
    )
    return selected

score_frames = []
for setup in SETUPS:
    print(f'Scoring {setup.name}...')
    score_frames.append(_model_scores_for_setup(setup))
scores = pd.concat(score_frames, ignore_index=True)
selected = select_model_attack_days(scores)

print('Giorni selezionati:')
print(selected.to_string(index=False))
days_per_setup_objective = selected.groupby(['setup', 'objective'])['attack_day'].nunique()
print('Numero di giorni per setup e obiettivo:')
print(days_per_setup_objective.to_string())
expected = pd.MultiIndex.from_product([
    [setup.name for setup in SETUPS], list(TARGET_SIGNALS),
], names=['setup', 'objective'])
missing = expected.difference(days_per_setup_objective.index)
incomplete = days_per_setup_objective.reindex(expected).dropna() < N_ATTACK_DAYS
if len(missing) or incomplete.any():
    raise RuntimeError(
        f'Selezione incompleta sopra soglia: mancanti={list(missing)}, '
        f'incompleti={days_per_setup_objective.reindex(expected)[incomplete].to_dict()}'
    )
scores.to_csv(RESULTS_DIR / 'attack_day_model_scores.csv', index=False)
selected.to_csv(RESULTS_DIR / 'attack_day_model_selected_days.csv', index=False)

print('Preparazione delle timing policy tradizionali...')
traditional_attack_day_plan = {}
for setup in SETUPS:
    train_mode = TRAIN_MODE_BY_SETUP.get(setup.name, 'cumulative')
    traditional_attack_day_plan.update(
        build_attack_day_plan(
            setups=[setup], policies=TIMING_POLICIES,
            model_folders=MODEL_FOLDERS, project_root=PROJECT_ROOT,
            data_folder=DATA_FOLDER, simulation_start=ATTACK_START,
            simulation_end=ATTACK_END, data_start=DATA_START,
            months=QUARTERLY_MONTHS, train_mode=train_mode,
            n_days=N_ATTACK_DAYS, sequence_length=SEQUENCE_LENGTH, min_day=30,
        )
    )
traditional_plan_df = pd.DataFrame([
    {'setup': setup, 'timing_policy': policy, 'attack_days': days.tolist()}
    for (setup, policy), days in traditional_attack_day_plan.items()
])
print(traditional_plan_df.to_string(index=False))

## Esecuzione del backtest

Per ogni setup vengono eseguiti sia il backtest attack-day sia i backtest con le timing policy tradizionali. I risultati vengono poi uniti nello stesso DataFrame; la probabilità del classificatore è valorizzata per i soli trial attack-day e tutto viene salvato in CSV.

In [ ]:
result_frames = []
# Backtest con il classificatore attack-day.
for setup in SETUPS:
    train_mode = TRAIN_MODE_BY_SETUP.get(setup.name, 'cumulative')
    setup_selected = selected[selected['setup'] == setup.name]
    for objective, objective_days in setup_selected.groupby('objective'):
        days = objective_days['attack_day'].to_numpy(dtype=int)
        if len(days) == 0:
            continue
        print(f'{setup.name} / {objective}: {days.tolist()}')
        result = run_targeted_attack_experiment(
            setups=[setup], attack_days=days, model_folders=MODEL_FOLDERS,
            data_folder=DATA_FOLDER, project_root=PROJECT_ROOT,
            target_signals={objective: TARGET_SIGNALS[objective]},
            sequence_length=SEQUENCE_LENGTH, max_abs_delta=MAX_ABS_DELTA,
            steps=GRID_STEPS, timing_policy='attack_day_model',
            simulation_start=ATTACK_START, simulation_end=ATTACK_END,
            data_start=DATA_START, months=QUARTERLY_MONTHS,
            eval_cr_period=EVALUATION_WINDOW_DAYS, train_mode=train_mode,
            verbose=True, diagnostics=True,
        )
        result_frames.append(result)

# Backtest con le timing policy tradizionali, usando lo stesso flusso del notebook di riferimento.
for setup in SETUPS:
    train_mode = TRAIN_MODE_BY_SETUP.get(setup.name, 'cumulative')
    for policy in TIMING_POLICIES:
        days = traditional_attack_day_plan[(setup.name, policy)]
        print(f'{setup.name} / {policy}: {days.tolist()}')
        result_frames.append(
            run_targeted_attack_experiment(
                setups=[setup], attack_days=days, model_folders=MODEL_FOLDERS,
                data_folder=DATA_FOLDER, project_root=PROJECT_ROOT,
                target_signals=TARGET_SIGNALS, sequence_length=SEQUENCE_LENGTH,
                max_abs_delta=MAX_ABS_DELTA, steps=GRID_STEPS,
                timing_policy=policy, simulation_start=ATTACK_START,
                simulation_end=ATTACK_END, data_start=DATA_START,
                months=QUARTERLY_MONTHS, eval_cr_period=EVALUATION_WINDOW_DAYS,
                train_mode=train_mode, verbose=True, diagnostics=True,
            )
        )

if not result_frames:
    raise RuntimeError('Il modello non ha selezionato alcun giorno sopra soglia.')
results = pd.concat(result_frames, ignore_index=True)
results = results.merge(
    scores[['setup', 'attack_day', 'objective', 'attack_probability']],
    on=['setup', 'attack_day', 'objective'], how='left',
)
results_path = RESULTS_DIR / 'attack_day_model_backtest.csv'
results.to_csv(results_path, index=False)
print(f'Risultati salvati in: {results_path}')

## Analisi dei risultati

In [ ]:
summary = compute_summary_by_setup_and_timing(results)
setup_summary = compute_summary_by_setup(results)
timing_summary, policy_summary = compute_timing_summaries(results)
print('=== Summary per setup ===')
display(setup_summary)
print('=== Summary modello / obiettivo ===')
display(summary)
print('=== Summary timing policy ===')
display(policy_summary)

plot_data = summary.copy()
plot_data['label'] = (
    plot_data['setup'] + ' / ' + plot_data['timing_policy'] + ' / ' + plot_data['objective']
)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(plot_data['label'], plot_data['nontrivial_attack_success_rate'], color='#2f6f73')
ax.set_ylabel('Non-trivial Attack Success Rate')
ax.set_xlabel('ATS setup / timing policy / objective')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=75, labelsize=8)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'attack_day_model_backtest_asr.png', dpi=150)
plt.show()